In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from torchinfo import summary
from of_transformer import OfTransformer
from simple_network import DumbNeuralNetwork
from net_utils import *
from data_utils import *
from plotting_utils import *

from sklearn.model_selection import train_test_split

import numpy as np
import uproot
import awkward as ak
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

In [ ]:
print(torch.cuda.is_available())
cuda_id = torch.cuda.current_device()
print(cuda_id)
print(torch.cuda.get_device_name(cuda_id))

Start by loading MC and PD files.

In [ ]:
# Load mc
f_mc = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_May19_MGPy8FxFxRew_syst_train.root')
tree_mc = f_mc['OmniTree']
tree_mc.show(name_width=50)

In [ ]:
f_pd = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_Aug5_PseudoDataSRew_Dec15.root')
tree_pd = f_pd['OmniTree']
tree_pd.show(name_width=50)

Next we need to build torch tensors from the data. One open question is how to handle the muon kinematics, since they are distinct from the tracks. For now I will include the muon information in the same way as the tracks, but this should be changed in the future. Probably the best way is to include a one-hot encoded input, which tells the network where the 3-vector is a muon, track from 1st jet, track from 2nd jet, etc.

In [ ]:
# Pass 190 flags
pass190_mc = ak.to_numpy(tree_mc['pass190'].array())
print("We have a fracion {} of good events in mc".format(np.sum(pass190_mc) / len(pass190_mc)))
pass190_pd = ak.to_numpy(tree_pd['pass190'].array())
print("We have a fracion {} of good events in pseudodata".format(np.sum(pass190_pd) / len(pass190_pd)))

In [ ]:
# Load MC kinematics
mc_kinematics, mc_mask = get_kinematics(tree_mc, filter=pass190_mc)
print(mc_kinematics.shape)
print(mc_mask.shape)


In [ ]:
# Load pseudodata kinematics
pd_kinematics, pd_mask = get_kinematics(tree_pd, filter=pass190_pd, max_tracks=mc_kinematics.shape[2]-2)
print(pd_kinematics.shape)
print(pd_mask.shape)

Next we need to load the weights. For MC this is easy. For pseudodata we need to take the missing track events into account.

In [ ]:
# MC weights
mc_weights = ak.to_numpy(tree_mc['weight'].array())
mc_weights = np.expand_dims(mc_weights[pass190_mc == 1], axis=1)
print(mc_weights.shape)

In [ ]:
print(np.mean(mc_weights))

In [ ]:
# Pseudodata weights
pd_weights = ak.to_numpy(tree_pd['weight'].array())
pd_weights = np.expand_dims(pd_weights[pass190_pd == 1], axis=1)

# Drop 11k weights at the end since we don't have tracks for those events
pd_weights = pd_weights[:pd_kinematics.shape[0]]

print(pd_weights.shape)

In [ ]:
bins = np.linspace(0, 2.5, 150)
plt.hist(mc_weights, bins=bins, alpha=0.5, label='MC', density=True)
plt.hist(pd_weights, bins=bins, alpha=0.5, label='Pseudodata', density=True)
add_stats_box(plt.gca(), mc_weights)
plt.legend(loc='center right')
plt.yscale('log')
plt.xlabel('Starting Weight')
plt.ylabel('A.U.')
plt.show()

In [ ]:
class_ratio = np.sum(mc_weights) / np.sum(pd_weights)
print("Class ratio: {}".format(class_ratio))

Make the labels. We will call pseudodata signal, and MC background.

In [ ]:
# Build labels
mc_labels = np.zeros((mc_kinematics.shape[0], 1), dtype=np.float32)
pd_labels = np.ones((pd_kinematics.shape[0], 1), dtype=np.float32)

Concatenate MC and PD together. Since with the weights the class ratio is nearly one, just use all of the events

In [ ]:
# Concatenate MC and pseudodata together
kinematics = np.concatenate([mc_kinematics, pd_kinematics], axis=0)
mask = np.concatenate([mc_mask, pd_mask], axis=0)
weights = np.concatenate([mc_weights, pd_weights], axis=0)
labels = np.concatenate([mc_labels, pd_labels], axis=0)

In [ ]:
print(mask[1,0,:])

Make the one-hot encoding dimensions which identify whether the object is a muon or a track

In [ ]:
# Make one-hot encoding identifying whether the object is a muon or a track
is_muon = np.concatenate([np.ones((kinematics.shape[0], 2)), np.zeros((kinematics.shape[0], kinematics.shape[2]-2))], axis=1)
is_track = np.concatenate([np.zeros((kinematics.shape[0], 2)), np.ones((kinematics.shape[0], kinematics.shape[2]-2))], axis=1)
one_hot = np.stack([is_muon, is_track], axis=1)
print(one_hot.shape)

Concatenate one-hot encoding dimensions onto the kinematics

In [ ]:
kinematics = np.concatenate([kinematics, one_hot], axis=1)

Finally done cleaning data. Make train test split and move data to GPU

In [ ]:
# Make train test split
kinematics_train, kinematics_test, labels_train, labels_test, mask_train, mask_test, weights_train, weights_test = train_test_split(kinematics, labels, mask, weights, test_size=0.2, random_state=42)
print(kinematics_train.shape, kinematics_test.shape, labels_train.shape, labels_test.shape, mask_train.shape, mask_test.shape, weights_train.shape, weights_test.shape)

In [ ]:
# Convert to torch tensors
device = 'cuda:0'
kinematics_train = torch.tensor(kinematics_train, dtype=torch.float32).to(device).contiguous()
kinematics_test = torch.tensor(kinematics_test, dtype=torch.float32).to(device).contiguous()
labels_train = torch.tensor(labels_train, dtype=torch.float32).to(device)
labels_test = torch.tensor(labels_test, dtype=torch.float32).to(device)
mask_train = torch.tensor(mask_train, dtype=torch.float32).to(device)
mask_test = torch.tensor(mask_test, dtype=torch.float32).to(device)
weights_train = torch.tensor(weights_train, dtype=torch.float32).to(device)
weights_test = torch.tensor(weights_test, dtype=torch.float32).to(device)

In [ ]:
# Build a pytorch dataset
train_dataset = torch.utils.data.TensorDataset(kinematics_train, labels_train, mask_train, weights_train)
test_dataset = torch.utils.data.TensorDataset(kinematics_test, labels_test, mask_test, weights_test)

In [ ]:
# Build pytorch data loaders
batch_size = 256
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

In [ ]:
print(len(train_loader))

Data is ready. Next we need to build a model!

In [ ]:
from of_transformer import OfTransformer
from simple_network import DumbNeuralNetwork
part = OfTransformer(
    5,
    num_classes=1,
    trim=False,
    embed_dims=[128, 128, 128],
    fc_params=[(256, 0.0)],
    pair_embed_dims=None,
    cls_block_params={'dropout': 0, 'attn_dropout': 0, 'activation_dropout': 0, 'num_heads': 32},
    num_cls_layers=1,
    block_params={'dropout': 0, 'attn_dropout': 0, 'activation_dropout': 0, 'num_heads': 32},
    num_layers=3
)
# part = DumbNeuralNetwork()
next(part.parameters()).device

In [ ]:
summary(part, input_shape=kinematics_train.shape[1:])

In [ ]:
# Copy model to GPU
part.to('cuda:0')
next(part.parameters()).device

In [ ]:
# Practice forward pass
first_batch = next(iter(train_loader))
first_event = first_batch[0]
first_mask = first_batch[2]
with torch.no_grad():
    out = part(first_event)
print(first_event[0,...])
print(out)


Now train the model. Define and optimizer and a loss function

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss(reduction='none')
optimizer = torch.optim.Adam(part.parameters(), lr=1e-4)

And train for a few epochs

In [ ]:
epochs = int(10)
part.train()

for epoch in range(epochs):  # loop over the dataset multiple times

    print("Epoch {}".format(epoch + 1))
    loop_obj = tqdm(enumerate(train_loader), total=len(train_loader))

    running_loss = 0.0
    for i, batch in loop_obj:

        # Unpack batch 
        batch_kinematics, batch_labels, batch_mask, batch_weights = batch

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass, compute loss, backward pass, optimizer step
        out = part(batch_kinematics, mask=batch_mask)
        loss = criterion(out, batch_labels)
        loss = loss * batch_weights
        loss.mean().backward()
        optimizer.step()

        # Print stats
        running_loss += loss.mean().item()
        if i % 100 == 99:    # print every 100 mini-batches
            # print(f'Epoch {epoch + 1}, mini-batch {i + 1}: loss {running_loss / 100:.3f}')
            loop_obj.set_postfix({'loss': running_loss / 100})
            running_loss = 0.0


In [ ]:
import itertools

# Print results on first batch
first_batch = next(itertools.islice(iter(train_loader), 0, None))
first_event = first_batch[0]
first_mask = first_batch[2]
first_labels = first_batch[1]
sig = torch.nn.Sigmoid()

part.eval()
with torch.no_grad():
    out = part(first_event)
print(first_event[0,...])
print(sig(out[:10]).detach().cpu().numpy().flatten())
print(first_labels[:10])


Now we need to determine whether there is a reasonable weighting derived by this network. Run prediction over testing set and plot the reweighting

In [ ]:
# Run prediction over the test dataloader
part.eval()
with torch.no_grad():
    predictions = []
    for batch in tqdm(test_loader):
        batch_kinematics, batch_labels, batch_mask, batch_weights = batch
        out = part(batch_kinematics, mask=batch_mask)
        predictions.append(out.detach().cpu().numpy().flatten())
    predictions = np.concatenate(predictions)
    probs = 1 / (1 + np.exp(-predictions))
    derived_weights = probs / (1 - probs)

In [ ]:
# Plot the quality of the reweighting
test_dataset = test_loader.dataset
test_events = test_dataset[:][0].detach().cpu().numpy()
test_weights = test_dataset[:][3].detach().cpu().numpy().flatten()

# Extract muon kinematics
alldata = {}
alldata['m1_pt'] = np.exp(test_events[:, 0, 0])
alldata['m2_pt'] = np.exp(test_events[:, 0, 1])
alldata['m1_eta'] = test_events[:, 1, 0]
alldata['m2_eta'] = test_events[:, 1, 1]
alldata['m1_phi'] = test_events[:, 2, 0]
alldata['m2_phi'] = test_events[:, 2, 1]
alldata['end_weights'] = test_weights * derived_weights
alldata['start_weights'] = test_weights
v1 = np.stack([alldata['m1_pt'] * np.cos(alldata['m1_phi']), alldata['m1_pt'] * np.sin(alldata['m1_phi']), alldata['m1_pt'] * np.sinh(alldata['m1_eta'])], axis=1)
v2 = np.stack([alldata['m2_pt'] * np.cos(alldata['m2_phi']), alldata['m2_pt'] * np.sin(alldata['m2_phi']), alldata['m2_pt'] * np.sinh(alldata['m2_eta'])], axis=1)
vall = v1 + v2
print(v1.shape)
print(vall.shape)
print(vall[:10,:])
alldata['mm_pt'] = np.sqrt(vall[:,0]**2 + vall[:,1]**2)

# Extract labels
labels = test_dataset[:][1].detach().cpu().numpy().flatten()

# Separate MC and pseudodata
mc_data = {key: val[labels == 0] for key, val in alldata.items()}
pd_data = {key: val[labels == 1] for key, val in alldata.items()}

In [ ]:
print(np.mean(mc_data['start_weights']))
print(np.mean(mc_data['end_weights']))
print(np.mean(pd_data['start_weights']))
print(np.mean(pd_data['end_weights']))

In [ ]:
from plotting_utils import *
fig = plot_reweighting(mc_data['m1_pt'], mc_data['start_weights'], mc_data['end_weights'], pd_data['m1_pt'], bins=np.linspace(0, 1e3, 200), xlabel=r'$p_{T, \mu_1}$ [GeV]', rlim=[0.95, 1.05])
fig.show()
fig = plot_reweighting(mc_data['m2_pt'], mc_data['start_weights'], mc_data['end_weights'], pd_data['m2_pt'], bins=np.linspace(0, 800, 200), xlabel=r'$p_{T, \mu_2}$ [GeV]')
fig.show()
fig = plot_reweighting(mc_data['m1_eta'], mc_data['start_weights'], mc_data['end_weights'], pd_data['m1_eta'], bins=np.linspace(-2.5, 2.5, 200), xlabel=r'$\eta_{\mu_1}$')
fig.show()
fig = plot_reweighting(mc_data['m2_eta'], mc_data['start_weights'], mc_data['end_weights'], pd_data['m2_eta'], bins=np.linspace(-2.5, 2.5, 200), xlabel=r'$\eta_{\mu_2}$')
fig.show()
fig = plot_reweighting(mc_data['m1_phi'], mc_data['start_weights'], mc_data['end_weights'], pd_data['m1_phi'], bins=np.linspace(-np.pi, np.pi, 200), xlabel=r'$\phi_{\mu_1}$', linear_scale=True, ylim=[0, 0.2])
fig.show()
fig = plot_reweighting(mc_data['m2_phi'], mc_data['start_weights'], mc_data['end_weights'], pd_data['m2_phi'], bins=np.linspace(-np.pi, np.pi, 200), xlabel=r'$\phi_{\mu_2}$', linear_scale=True, ylim=[0, 0.2])
fig.show()
fig = plot_reweighting(mc_data['mm_pt'], mc_data['start_weights'], mc_data['end_weights'], pd_data['mm_pt'], bins=np.linspace(0, 1e3, 200), xlabel=r'$p_{T, \mu\mu}$ [GeV]')
fig.show()

Lovely. We're getting all 6 dimensions plus one derived dimension with acceptable precision for now. This is great. Let's try training and evaluating something like this in production mode, and logging these plots during training. That way I can look for any overfitting effects.